In [1]:
# ======================
# Imports
# ======================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ======================
# Paths (عدل حسب مكان ملفاتك)
# ======================
TRAIN_FILE = "/kaggle/input/datasets/organizations/uciml/human-activity-recognition-with-smartphones/train.csv"  # مثال
TEST_FILE  = "/kaggle/input/datasets/organizations/uciml/human-activity-recognition-with-smartphones/test.csv"   # مثال

# ======================
# Dataset
# ======================
class HARDS(Dataset):
    def __init__(self, csv_file, label_encoder=None):
        self.data = pd.read_csv(csv_file)
        print(f"Columns in {csv_file}: {self.data.shape[1]}")

        # آخر عمود = label
        self.X = self.data.iloc[:, :-1].values.astype(np.float32)
        self.y_raw = self.data.iloc[:, -1].values

        # Encode labels
        if label_encoder is None:
            self.label_encoder = LabelEncoder()
            self.y = self.label_encoder.fit_transform(self.y_raw)
        else:
            self.label_encoder = label_encoder
            self.y = self.label_encoder.transform(self.y_raw)

        # نفترض كل صف = sequence واحد
        self.seq_len = 1
        self.X = self.X.reshape(-1, self.seq_len, self.X.shape[1])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# ======================
# Load Datasets
# ======================
train_dataset = HARDS(TRAIN_FILE)
test_dataset  = HARDS(TEST_FILE, label_encoder=train_dataset.label_encoder)  # نفس encoder للـ test

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ======================
# Model
# ======================
class LSTM_HAR(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape = [batch, seq_len, features]
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])  # ناخد آخر خطوة زمنية
        return out

input_size = train_dataset.X.shape[2]   # عدد الأعمدة (features)
hidden_size = 128
num_classes = len(train_dataset.label_encoder.classes_)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTM_HAR(input_size, hidden_size, num_classes).to(device)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ======================
# Training
# ======================
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# ======================
# Evaluation
# ======================
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Columns in /kaggle/input/datasets/organizations/uciml/human-activity-recognition-with-smartphones/train.csv: 563
Columns in /kaggle/input/datasets/organizations/uciml/human-activity-recognition-with-smartphones/test.csv: 563
Epoch [1/5], Loss: 0.5332
Epoch [2/5], Loss: 0.1614
Epoch [3/5], Loss: 0.1082
Epoch [4/5], Loss: 0.0830
Epoch [5/5], Loss: 0.0649
Test Accuracy: 91.41%
